In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Advanced ML libraries
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score, KFold, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_selection import SelectKBest, f_regression

# Models to test
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor, StackingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Feature importance and visualization
from sklearn.inspection import permutation_importance
import shap

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

# List input files
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# =====================================================
# STEP 1: DATA LOADING AND INITIAL EXPLORATION
# =====================================================

train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

print(f"Training set shape: {train.shape}")
print(f"Test set shape: {test.shape}")

# Save test IDs for submission
test_ID = test['Id']

# Display basic info about the dataset
print("\n=== DATASET OVERVIEW ===")
print(f"Number of features: {train.shape[1] - 1}")
print(f"Target variable: SalePrice")
print(f"Training samples: {train.shape[0]}")
print(f"Test samples: {test.shape[0]}")

# =====================================================
# STEP 2: COMPREHENSIVE EDA AND VISUALIZATION
# =====================================================

print("\n=== EXPLORATORY DATA ANALYSIS ===")

# Target variable analysis
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Original distribution
axes[0].hist(train['SalePrice'], bins=50, alpha=0.7, color='skyblue', edgecolor='black')
axes[0].set_title('Original SalePrice Distribution')
axes[0].set_xlabel('SalePrice')
axes[0].set_ylabel('Frequency')

# Log-transformed distribution
log_prices = np.log1p(train['SalePrice'])
axes[1].hist(log_prices, bins=50, alpha=0.7, color='lightgreen', edgecolor='black')
axes[1].set_title('Log-transformed SalePrice Distribution')
axes[1].set_xlabel('log(SalePrice)')
axes[1].set_ylabel('Frequency')

# Q-Q plot for normality check
from scipy import stats
stats.probplot(log_prices, dist="norm", plot=axes[2])
axes[2].set_title('Q-Q Plot (Log-transformed)')

plt.tight_layout()
plt.show()

# Basic statistics
print(f"\nSalePrice Statistics:")
print(f"Mean: ${train['SalePrice'].mean():,.2f}")
print(f"Median: ${train['SalePrice'].median():,.2f}")
print(f"Std: ${train['SalePrice'].std():,.2f}")
print(f"Skewness: {train['SalePrice'].skew():.3f}")
print(f"Skewness after log transform: {log_prices.skew():.3f}")

# Missing values analysis
def analyze_missing_values(df, name):
    missing = df.isnull().sum()
    missing_percent = (missing / len(df)) * 100
    missing_df = pd.DataFrame({
        'Feature': missing.index,
        'Missing_Count': missing.values,
        'Missing_Percent': missing_percent.values
    })
    missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Percent', ascending=False)
    
    if len(missing_df) > 0:
        print(f"\n=== MISSING VALUES IN {name.upper()} ===")
        print(missing_df.head(15))
        
        # Visualize missing values
        plt.figure(figsize=(12, 8))
        if len(missing_df) > 20:
            missing_df_plot = missing_df.head(20)
        else:
            missing_df_plot = missing_df
            
        plt.barh(range(len(missing_df_plot)), missing_df_plot['Missing_Percent'])
        plt.yticks(range(len(missing_df_plot)), missing_df_plot['Feature'])
        plt.xlabel('Missing Percentage (%)')
        plt.title(f'Missing Values in {name} Dataset')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()
    else:
        print(f"\n✅ No missing values in {name} dataset!")
    
    return missing_df

# Analyze missing values
missing_train = analyze_missing_values(train, 'Training')
missing_test = analyze_missing_values(test, 'Test')

# Correlation analysis with target
print("\n=== TOP CORRELATIONS WITH SALEPRICE ===")
correlations = train.select_dtypes(include=[np.number]).corrwith(train['SalePrice']).abs().sort_values(ascending=False)
print(correlations.head(15))

# Visualize top correlations
plt.figure(figsize=(10, 8))
top_corr = correlations.head(15)
colors = ['red' if x == 'SalePrice' else 'skyblue' for x in top_corr.index]
plt.barh(range(len(top_corr)), top_corr.values, color=colors)
plt.yticks(range(len(top_corr)), top_corr.index)
plt.xlabel('Absolute Correlation with SalePrice')
plt.title('Top 15 Features Correlated with SalePrice')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# =====================================================
# STEP 3: ADVANCED DATA CLEANING AND PREPROCESSING
# =====================================================

print("\n=== DATA CLEANING AND PREPROCESSING ===")

# Combine train and test for consistent preprocessing
y = train['SalePrice']
train_len = len(train)
all_data = pd.concat([train.drop(['SalePrice'], axis=1), test], axis=0, ignore_index=True)

print(f"Combined dataset shape: {all_data.shape}")

# Handle missing values strategically
def handle_missing_values(df):
    """
    Handle missing values based on domain knowledge and data exploration
    """
    df = df.copy()
    
    # Features where NaN means 'None' or 'Not Applicable'
    none_features = [
        'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
        'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
        'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
        'MasVnrType'
    ]
    
    for feature in none_features:
        if feature in df.columns:
            df[feature] = df[feature].fillna('None')
    
    # Numeric features where NaN means 0
    zero_features = [
        'GarageYrBlt', 'GarageArea', 'GarageCars',
        'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
        'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea'
    ]
    
    for feature in zero_features:
        if feature in df.columns:
            df[feature] = df[feature].fillna(0)
    
    # LotFrontage: fill with median by Neighborhood
    if 'LotFrontage' in df.columns:
        df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(
            lambda x: x.fillna(x.median())
        )
    
    # Remaining missing values
    # Numeric: fill with median
    numeric_features = df.select_dtypes(include=[np.number]).columns
    for feature in numeric_features:
        if df[feature].isnull().any():
            df[feature] = df[feature].fillna(df[feature].median())
    
    # Categorical: fill with mode
    categorical_features = df.select_dtypes(include=[object]).columns
    for feature in categorical_features:
        if df[feature].isnull().any():
            df[feature] = df[feature].fillna(df[feature].mode()[0])
    
    return df

# Apply missing value handling
all_data_cleaned = handle_missing_values(all_data)
print(f"Missing values after cleaning: {all_data_cleaned.isnull().sum().sum()}")

# =====================================================
# STEP 4: ADVANCED FEATURE ENGINEERING
# =====================================================

print("\n=== FEATURE ENGINEERING ===")

def create_advanced_features(df):
    """
    Create advanced features based on domain knowledge and feature interactions
    """
    df = df.copy()
    
    # Total square footage features
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['TotalBathrooms'] = (df['FullBath'] + (0.5 * df['HalfBath']) + 
                           df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath']))
    df['TotalPorchSF'] = (df['OpenPorchSF'] + df['3SsnPorch'] + 
                         df['EnclosedPorch'] + df['ScreenPorch'] + df['WoodDeckSF'])
    
    # Age-related features
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['YearsSinceRemodel'] = df['YrSold'] - df['YearRemodAdd']
    
    # Quality-area interactions
    df['OverallQual_x_GrLivArea'] = df['OverallQual'] * df['GrLivArea']
    df['OverallQual_x_TotalSF'] = df['OverallQual'] * df['TotalSF']
    df['BsmtQual_x_TotalBsmtSF'] = df['BsmtQual'].map({'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}) * df['TotalBsmtSF']
    
    # Garage features
    df['HasGarage'] = (df['GarageType'] != 'None').astype(int)
    df['HasBasement'] = (df['TotalBsmtSF'] > 0).astype(int)
    df['HasPool'] = (df['PoolArea'] > 0).astype(int)
    df['HasFireplace'] = (df['Fireplaces'] > 0).astype(int)
    
    # Price per square foot proxies
    df['LotArea_x_OverallQual'] = df['LotArea'] * df['OverallQual']
    
    # Neighborhood quality (will be calculated later)
    # This is a placeholder for target encoding
    
    return df

# Apply feature engineering
all_data_featured = create_advanced_features(all_data_cleaned)
print(f"Dataset shape after feature engineering: {all_data_featured.shape}")

# Handle categorical variables
def encode_categorical_features(df):
    """
    Encode categorical features using various methods
    """
    df = df.copy()
    
    # Ordinal encoding for quality features
    quality_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
    quality_features = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 
                       'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']
    
    for feature in quality_features:
        if feature in df.columns:
            df[feature] = df[feature].map(quality_map)
    
    # Ordinal encoding for other features
    basement_map = {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4}
    if 'BsmtExposure' in df.columns:
        df['BsmtExposure'] = df['BsmtExposure'].map(basement_map)
    
    slope_map = {'Sev': 1, 'Mod': 2, 'Gtl': 3}
    if 'LandSlope' in df.columns:
        df['LandSlope'] = df['LandSlope'].map(slope_map)
    
    # One-hot encoding for remaining categorical features
    categorical_features = df.select_dtypes(include=[object]).columns
    df = pd.get_dummies(df, columns=categorical_features, drop_first=True)
    
    return df

# Apply categorical encoding
all_data_encoded = encode_categorical_features(all_data_featured)
print(f"Dataset shape after encoding: {all_data_encoded.shape}")

# =====================================================
# STEP 5: OUTLIER DETECTION AND TREATMENT
# =====================================================

print("\n=== OUTLIER DETECTION ===")

# Split back to train and test for outlier detection
X_train_full = all_data_encoded[:train_len]
X_test_full = all_data_encoded[train_len:]

# Detect outliers using IQR method for target variable
Q1 = y.quantile(0.25)
Q3 = y.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = y[(y < lower_bound) | (y > upper_bound)]
print(f"Found {len(outliers)} outliers in target variable")
print(f"Outlier range: < ${lower_bound:,.0f} or > ${upper_bound:,.0f}")

# Visualize outliers
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.boxplot(y)
plt.title('SalePrice Boxplot')
plt.ylabel('SalePrice')

plt.subplot(1, 3, 2)
plt.scatter(X_train_full['GrLivArea'], y, alpha=0.5)
plt.xlabel('GrLivArea')
plt.ylabel('SalePrice')
plt.title('SalePrice vs GrLivArea')

plt.subplot(1, 3, 3)
plt.scatter(X_train_full['TotalSF'], y, alpha=0.5)
plt.xlabel('TotalSF')
plt.ylabel('SalePrice')
plt.title('SalePrice vs TotalSF')

plt.tight_layout()
plt.show()

# Remove extreme outliers (optional - be careful with this)
# For competition, we'll keep all data but could remove obvious errors
extreme_outliers = X_train_full[(X_train_full['GrLivArea'] > 4000) & (y < 300000)].index
print(f"Extreme outliers to potentially remove: {len(extreme_outliers)}")

# =====================================================
# STEP 6: PREPARE FINAL DATASETS
# =====================================================

# Log transform target (as required by competition metric)
y_log = np.log1p(y)

# Final datasets
X_train = X_train_full
X_test = X_test_full
y_train = y_log

print(f"Final training shape: {X_train.shape}")
print(f"Final test shape: {X_test.shape}")

# =====================================================
# STEP 7: MODEL SELECTION AND TRAINING
# =====================================================

print("\n=== MODEL TRAINING AND COMPARISON ===")

# Cross-validation setup
cv = KFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_model(model, X, y, cv):
    """Evaluate model using cross-validation"""
    scores = cross_val_score(model, X, y, scoring='neg_mean_squared_error', cv=cv)
    rmse_scores = np.sqrt(-scores)
    return rmse_scores.mean(), rmse_scores.std()

# Define models to test
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=10.0),
    'Lasso': Lasso(alpha=0.0005, max_iter=10000),
    'ElasticNet': ElasticNet(alpha=0.0005, l1_ratio=0.9, max_iter=10000),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, eval_metric='rmse'),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42, verbose=-1),
    'CatBoost': CatBoostRegressor(n_estimators=100, random_state=42, verbose=False),
    'Neural Network': MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42)
}

# Evaluate all models
results = {}
print("Evaluating models...")

for name, model in models.items():
    print(f"Training {name}...")
    try:
        mean_rmse, std_rmse = evaluate_model(model, X_train, y_train, cv)
        results[name] = {'RMSE': mean_rmse, 'Std': std_rmse}
        print(f"{name}: RMSE = {mean_rmse:.5f} (+/- {std_rmse:.5f})")
    except Exception as e:
        print(f"Error with {name}: {e}")
        results[name] = {'RMSE': float('inf'), 'Std': 0}

# Create results DataFrame
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('RMSE')

print("\n=== MODEL COMPARISON RESULTS ===")
print(results_df)

# Visualize model comparison
plt.figure(figsize=(12, 8))
x_pos = range(len(results_df))
plt.barh(x_pos, results_df['RMSE'], xerr=results_df['Std'], alpha=0.7)
plt.yticks(x_pos, results_df.index)
plt.xlabel('RMSE (Cross-Validation)')
plt.title('Model Performance Comparison')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# =====================================================
# STEP 8: HYPERPARAMETER TUNING FOR BEST MODELS
# =====================================================

print("\n=== HYPERPARAMETER TUNING ===")

# Get top 3 models for tuning
top_models = results_df.head(3)
print(f"Top 3 models for hyperparameter tuning:")
print(top_models)

# Hyperparameter grids
param_grids = {
    'XGBoost': {
        'n_estimators': [1000, 2000],
        'max_depth': [3, 4, 5],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample': [0.8, 0.9]
    },
    'LightGBM': {
        'n_estimators': [1000, 2000],
        'max_depth': [3, 4, 5],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample': [0.8, 0.9]
    },
    'Random Forest': {
        'n_estimators': [500, 1000],
        'max_depth': [10, 15, 20],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    },
    'Gradient Boosting': {
        'n_estimators': [500, 1000],
        'max_depth': [3, 4, 5],
        'learning_rate': [0.01, 0.05, 0.1]
    }
}

# Tune best model
best_model_name = results_df.index[0]
tuned_models = {}

if best_model_name in param_grids:
    print(f"\nTuning {best_model_name}...")
    
    if best_model_name == 'XGBoost':
        base_model = XGBRegressor(random_state=42, eval_metric='rmse')
    elif best_model_name == 'LightGBM':
        base_model = LGBMRegressor(random_state=42, verbose=-1)
    elif best_model_name == 'Random Forest':
        base_model = RandomForestRegressor(random_state=42)
    elif best_model_name == 'Gradient Boosting':
        base_model = GradientBoostingRegressor(random_state=42)
    
    # Use RandomizedSearchCV for efficiency
    grid_search = RandomizedSearchCV(
        base_model,
        param_grids[best_model_name],
        n_iter=20,
        cv=cv,
        scoring='neg_mean_squared_error',
        random_state=42,
        n_jobs=-1
    )
    
    grid_search.fit(X_train, y_train)
    
    tuned_models[best_model_name] = grid_search.best_estimator_
    tuned_rmse = np.sqrt(-grid_search.best_score_)
    
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best RMSE: {tuned_rmse:.5f}")
    print(f"Improvement: {results_df.loc[best_model_name, 'RMSE'] - tuned_rmse:.5f}")

# =====================================================
# STEP 9: ENSEMBLE METHODS
# =====================================================

print("\n=== ENSEMBLE METHODS ===")

# Select best individual models for ensemble
best_individual_models = [
    ('xgb', XGBRegressor(n_estimators=1000, max_depth=4, learning_rate=0.05, random_state=42, eval_metric='rmse')),
    ('lgb', LGBMRegressor(n_estimators=1000, max_depth=4, learning_rate=0.05, random_state=42, verbose=-1)),
    ('rf', RandomForestRegressor(n_estimators=500, max_depth=15, random_state=42)),
    ('gb', GradientBoostingRegressor(n_estimators=500, max_depth=4, learning_rate=0.05, random_state=42))
]

# Voting Regressor
voting_regressor = VotingRegressor(best_individual_models)

# Stacking Regressor
stacking_regressor = StackingRegressor(
    estimators=best_individual_models,
    final_estimator=Ridge(alpha=10.0),
    cv=cv
)

# Evaluate ensemble methods
ensemble_models = {
    'Voting Regressor': voting_regressor,
    'Stacking Regressor': stacking_regressor
}

ensemble_results = {}
for name, model in ensemble_models.items():
    print(f"Training {name}...")
    mean_rmse, std_rmse = evaluate_model(model, X_train, y_train, cv)
    ensemble_results[name] = {'RMSE': mean_rmse, 'Std': std_rmse}
    print(f"{name}: RMSE = {mean_rmse:.5f} (+/- {std_rmse:.5f})")

# =====================================================
# STEP 10: FEATURE IMPORTANCE ANALYSIS
# =====================================================

print("\n=== FEATURE IMPORTANCE ANALYSIS ===")

# Train best model for feature importance
if best_model_name in tuned_models:
    best_model = tuned_models[best_model_name]
else:
    best_model = models[best_model_name]

best_model.fit(X_train, y_train)

# Get feature importance
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': X_train.columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    # Plot top 10 feature importance
    plt.figure(figsize=(12, 8))
    top_features = feature_importance.head(10)
    plt.barh(range(len(top_features)), top_features['importance'])
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Feature Importance')
    plt.title(f'Top 10 Feature Importance - {best_model_name}')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    print("Top 10 Most Important Features:")
    print(feature_importance.head(10))

# =====================================================
# STEP 11: FINAL MODEL SELECTION AND PREDICTION
# =====================================================

print("\n=== FINAL MODEL SELECTION ===")

# Choose the best performing model
all_results = {**results, **ensemble_results}
best_final_model_name = min(all_results.keys(), key=lambda k: all_results[k]['RMSE'])
best_final_rmse = all_results[best_final_model_name]['RMSE']

print(f"Best model: {best_final_model_name}")
print(f"Best RMSE: {best_final_rmse:.5f}")

# Train final model
if best_final_model_name in ensemble_models:
    final_model = ensemble_models[best_final_model_name]
elif best_final_model_name in tuned_models:
    final_model = tuned_models[best_final_model_name]
else:
    final_model = models[best_final_model_name]

print("Training final model on full dataset...")
final_model.fit(X_train, y_train)

# Make predictions
print("Making predictions...")
y_pred_log = final_model.predict(X_test)
y_pred = np.expm1(y_pred_log)  # Convert back from log space

# =====================================================
# STEP 12: SUBMISSION FILE GENERATION
# =====================================================

print("\n=== GENERATING SUBMISSION ===")

# Create submission
submission = pd.DataFrame({
    'Id': test_ID,
    'SalePrice': y_pred
})

# Save submission file
submission.to_csv('/kaggle/working/submission.csv', index=False)
print("✅ Submission file created: submission.csv")

# Display submission statistics
print(f"\nSubmission Statistics:")
print(f"Predictions count: {len(submission)}")
print(f"Mean predicted price: ${submission['SalePrice'].mean():,.2f}")
print(f"Median predicted price: ${submission['SalePrice'].median():,.2f}")
print(f"Min predicted price: ${submission['SalePrice'].min():,.2f}")
print(f"Max predicted price: ${submission['SalePrice'].max():,.2f}")

print(submission.head(10))

# =====================================================
# STEP 13: MODEL PERFORMANCE VISUALIZATION
# =====================================================

print("\n=== FINAL VISUALIZATIONS ===")

# Prediction vs actual on training set
train_pred_log = final_model.predict(X_train)
train_pred = np.expm1(train_pred_log)

plt.figure(figsize=(15, 5))

# Actual vs Predicted
plt.subplot(1, 3, 1)
plt.scatter(y, train_pred, alpha=0.5)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
plt.xlabel('Actual SalePrice')
plt.ylabel('Predicted SalePrice')
plt.title('Actual vs Predicted (Training Set)')

# Residuals plot
plt.subplot(1, 3, 2)
residuals = y - train_pred
plt.scatter(train_pred, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted SalePrice')
plt.ylabel('Residuals')
plt.title('Residuals Plot')

# Model comparison final chart
plt.subplot(1, 3, 3)
model_names = list(all_results.keys())
model_rmses = [all_results[name]['RMSE'] for name in model_names]
colors = ['red' if name == best_final_model_name else 'skyblue' for name in model_names]
plt.barh(range(len(model_names)), model_rmses, color=colors)
plt.yticks(range(len(model_names)), model_names)
plt.xlabel('RMSE')
plt.title('Final Model Comparison')
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

# Training set performance metrics
r2 = r2_score(y, train_pred)
rmse_train = np.sqrt(mean_squared_error(y, train_pred))

print(f"\n=== FINAL MODEL PERFORMANCE ===")
print(f"Model: {best_final_model_name}")
print(f"Training R²: {r2:.4f}")
print(f"Training RMSE: ${rmse_train:,.2f}")
print(f"Cross-validation RMSE: {best_final_rmse:.5f}")
